# OWS reference model
1. import `config_tool.py`
2. choose `city`
3. import `config_project.py`
4. define `cwd_data_meta`, `cwd_data_str`, and the other project folders
5. read the files directly in the notebook
6. pass the loaded notebook variables to the v4 script for the heavy processing

Edit the `city`, input-file paths, and options in the notebook cells below. You should not need to edit the script for normal runs.


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 140)


## Project configuration


In [ ]:
cwd = os.getcwd()
cwd_main = os.path.abspath(os.path.join(cwd, os.pardir))
os.chdir(cwd_main)
print("cwd_main:", cwd_main)


import config_tool as cfm


In [ ]:
city = os.environ.get("QC_PROJECT_ID", "project_id")


## Import input data of the project — `config_project.py`


In [ ]:

cwd_project = Path(cfm.cwd_data) / city


os.chdir(cwd_project)
print("cwd_project:", cwd_project)


sys.modules.pop("config_project", None)
import config_project as cfp


In [ ]:

cwd_data_str = Path(cfp.cwd_data_str)
cwd_data_meta = Path(cfp.cwd_data_meta)
cwd_data_qc = Path(cfp.cwd_data_qc) if hasattr(cfp, "cwd_data_qc") else None
cwd_data_qc_results = Path(cfp.cwd_results_qc)

benchmark_dir = cwd_data_qc_results / "qc_benchmark"
reference_dir = benchmark_dir / "ows_reference"
reference_dir.mkdir(parents=True, exist_ok=True)

print("cwd_data_str:", cwd_data_str)
print("cwd_data_meta:", cwd_data_meta)
print("cwd_data_qc_results:", cwd_data_qc_results)
print("reference_dir:", reference_dir)


In [ ]:

print("city:", cfp.city)
print("first date:", cfp.first_date)
print("last date:", cfp.last_date)
print("lat,long:", cfp.lat, cfp.long)
print("plot:", cfp.plot)

start_date = pd.to_datetime(cfp.first_date, format="%d-%m-%Y %H:%M")
end_date = pd.to_datetime(cfp.last_date, format="%d-%m-%Y %H:%M")
year_span = f"{start_date.year}-{end_date.year}"

print("data from", start_date, "to", end_date)
print("year_span:", year_span)


## Import benchmark helper and v4 reference script


In [ ]:
os.chdir(cfm.cwd_scripts_preprocesing)
if str(Path(cfm.cwd_scripts_preprocesing)) not in sys.path:
    sys.path.insert(0, str(Path(cfm.cwd_scripts_preprocesing)))


import qc_benchmark_helpers_ows_patch as qcb


import qc_ows_reference_model_v4_simple as owsref

print("qcb:", qcb.__file__)
print("owsref:", owsref.__file__)


## Read hourly station data and metadata

Edit the paths in this cell if you want to use different input files. The variables are passed directly to the script later.


In [ ]:

CWS_coordinates_wunder_file = cwd_data_meta / f"Coordinates_{cfp.city}_CWS_Wunderground_str_all.csv"
CWS_ta_wunder_file = cwd_data_str / f"ta_{cfp.city}_{year_span}_h_CWS_Wunderground_str.csv"

CWS_coordinates_wunder = pd.read_csv(CWS_coordinates_wunder_file)
CWS_ta_wunder = pd.read_csv(
    CWS_ta_wunder_file,
    index_col="date",
    parse_dates=True,
)


CWS_coordinates_net_file = cwd_data_meta / f"Coordinates_{cfp.city}_CWS_Netatmo_str_all.csv"
CWS_ta_net_file = cwd_data_str / f"ta_{cfp.city}_{year_span}_h_CWS_Netatmo_str.csv"

CWS_coordinates_net = pd.read_csv(CWS_coordinates_net_file)
CWS_ta_net = pd.read_csv(
    CWS_ta_net_file,
    index_col="date",
    parse_dates=True,
)


OWS_coordinates_file = cwd_data_meta / f"Coordinates_{cfp.city}_OWS_str_all.csv"
OWS_ta_file = cwd_data_str / f"ta_{cfp.city}_{year_span}_h_OWS_str.csv"

OWS_coordinates = pd.read_csv(OWS_coordinates_file)
OWS_ta = pd.read_csv(
    OWS_ta_file,
    index_col="date",
    parse_dates=True,
)

print("CWS_ta_net:", CWS_ta_net.shape)
print("CWS_ta_wunder:", CWS_ta_wunder.shape)
print("OWS_ta:", OWS_ta.shape)
display(CWS_coordinates_net.head())
display(OWS_coordinates.head())


## Read optional ERA5, LST, and NDVI covariates

Edit any path in this cell if needed. Missing optional files are skipped.


In [ ]:
folder_era5 = cwd_data_str / "ERA5Land_hourly"
folder_lst = cwd_data_str / "LST_hourly"
folder_ndvi = cwd_data_str / "NDVI_S2"

def maybe_read_csv(path):
    path = Path(path)
    if path.exists():
        print("Reading:", path)
        return pd.read_csv(path)
    print("Missing optional file:", path)
    return None


era5_net_file = folder_era5 / "CWS_Netatmo" / f"ERA5Land_hourly_{cfp.city}_{year_span}_CWS_Netatmo_str_long.csv"
era5_wunder_file = folder_era5 / "CWS_Wunderground" / f"ERA5Land_hourly_{cfp.city}_{year_span}_CWS_Wunderground_str_long.csv"
era5_ows_file = folder_era5 / "OWS" / f"ERA5Land_hourly_{cfp.city}_{year_span}_OWS_str_long.csv"

era5_net = maybe_read_csv(era5_net_file)
era5_wunder = maybe_read_csv(era5_wunder_file)
era5_ows = maybe_read_csv(era5_ows_file)


lst_net_file = folder_lst / f"LST_hourly_{cfp.city}_CWS_Netatmo_long.csv"
lst_wunder_file = folder_lst / f"LST_hourly_{cfp.city}_CWS_Wunderground_long.csv"
lst_ows_file = folder_lst / f"LST_hourly_{cfp.city}_OWS_long.csv"

lst_net = maybe_read_csv(lst_net_file)
lst_wunder = maybe_read_csv(lst_wunder_file)
lst_ows = maybe_read_csv(lst_ows_file)


ndvi_net_file = folder_ndvi / f"NDVI_S2_{cfp.city}_CWS_Netatmo_long.csv"
ndvi_wunder_file = folder_ndvi / f"NDVI_S2_{cfp.city}_CWS_Wunderground_long.csv"
ndvi_ows_file = folder_ndvi / f"NDVI_S2_{cfp.city}_OWS_long.csv"

ndvi_net = maybe_read_csv(ndvi_net_file)
ndvi_wunder = maybe_read_csv(ndvi_wunder_file)
ndvi_ows = maybe_read_csv(ndvi_ows_file)


## Package the notebook inputs for the script

This is just a compact handoff from the notebook to the v4 script. The script still does the processing/modeling in the background.


In [ ]:
raw = {
    "year_span": year_span,
    "input_paths": {
        "CWS_coordinates_wunder_file": str(CWS_coordinates_wunder_file),
        "CWS_ta_wunder_file": str(CWS_ta_wunder_file),
        "CWS_coordinates_net_file": str(CWS_coordinates_net_file),
        "CWS_ta_net_file": str(CWS_ta_net_file),
        "OWS_coordinates_file": str(OWS_coordinates_file),
        "OWS_ta_file": str(OWS_ta_file),
    },
    "CWS_coordinates_wunder": CWS_coordinates_wunder,
    "CWS_ta_wunder": CWS_ta_wunder,
    "CWS_coordinates_net": CWS_coordinates_net,
    "CWS_ta_net": CWS_ta_net,
    "OWS_coordinates": OWS_coordinates,
    "OWS_ta": OWS_ta,
}

env_raw = {
    "era5_net": era5_net,
    "era5_wunder": era5_wunder,
    "era5_ows": era5_ows,
    "lst_net": lst_net,
    "lst_wunder": lst_wunder,
    "lst_ows": lst_ows,
    "ndvi_net": ndvi_net,
    "ndvi_wunder": ndvi_wunder,
    "ndvi_ows": ndvi_ows,
    "optional_paths": {
        "era5_net_file": str(era5_net_file),
        "era5_wunder_file": str(era5_wunder_file),
        "era5_ows_file": str(era5_ows_file),
        "lst_net_file": str(lst_net_file),
        "lst_wunder_file": str(lst_wunder_file),
        "lst_ows_file": str(lst_ows_file),
        "ndvi_net_file": str(ndvi_net_file),
        "ndvi_wunder_file": str(ndvi_wunder_file),
        "ndvi_ows_file": str(ndvi_ows_file),
    },
}

print("Notebook input handoff is ready.")


## Options


In [ ]:
config = owsref.ReferenceModelConfig(
    city=city,


    output_subdir="ows_reference",
    run_label="catboost_iteration01",
    overwrite_existing_run=False,
    rebuild=False,
    random_state=42,
    chunk_freq="M",


    reference_method="catboost",
    run_catboost_regressor=True,


    k_nearest_ows=8,
    ows_radius_km=8.0,
    fallback_to_k_nearest=True,
    idw_power=2.0,


    valid_start="2021-10-01",
    test_start="2021-11-01",
    calibration_mode="time_train",
    save_both_calibration_modes=True,
    abnormal_quantile=0.995,
    ambiguous_lower_quantile=0.95,
    sigma_floor_c=0.20,
    sigma_min_group_n=200,


    lst_asof_tolerance="3h",
    lst_asof_direction="nearest",
    ndvi_asof_tolerance="21D",
    ndvi_asof_direction="backward",
    use_dynamic_env_columns=True,
    use_ows_satellite_context=True,


    catboost_cv_mode="group_kfold",
    catboost_n_splits=5,
    catboost_iterations=1200,
    catboost_learning_rate=0.04,
    catboost_depth=8,
    catboost_thread_count=15,
    catboost_used_ram_limit="42gb",
    catboost_use_gpu=False,
)

print(config)


## Run the v4 pipeline

The notebook variables above are passed to the script, so the script does not need file-path edits.


In [ ]:
manifest = owsref.run_reference_pipeline(
    config,
    cfm=cfm,
    cfp=cfp,
    project_dir=cwd_project,
    raw=raw,
    env_raw=env_raw,
)

print("Output dir:", manifest["output_dir"])
print("Primary OWS predictions:", manifest["ows_primary_reference_path"])
print("Primary CWS residual features:", manifest["cws_primary_reference_path"])
print("Primary retention curve:", manifest["primary_retention_curve_path"])


## Inspect key outputs


In [ ]:
print("Output dir:", manifest["output_dir"])
print("Coverage audit:", manifest["station_coverage_audit_path"])
print("Primary OWS predictions:", manifest["ows_primary_reference_path"])
print("Primary CWS residual features:", manifest["cws_primary_reference_path"])
print("Primary retention curve:", manifest["primary_retention_curve_path"])

print("\nThresholds by method/mode:")
manifest["thresholds_by_method_mode"]


## Coverage audit

Use this to confirm the OWS metadata/environment files cover the expected station universe before interpreting model results.


In [ ]:
coverage = pd.read_csv(manifest["station_coverage_audit_path"])
coverage


## Compare calibration modes


In [ ]:
for key, paths in manifest["calibration_paths_by_method_mode"].items():
    print("\n", key)
    print(" json:", paths.get("json"))
    if paths.get("json"):
        with open(paths["json"]) as f:
            cal = json.load(f)
        print(" global_sigma:", cal.get("global_sigma"))
        print(" thresholds:", cal.get("thresholds"))


## Read OWS summaries


In [ ]:
primary_key = f"{manifest['selected_reference_method']}__{manifest['primary_calibration_mode']}"
summary_paths = manifest["ows_summary_paths_by_method_mode"][primary_key]

overall = pd.read_csv(summary_paths["overall"])
by_hour = pd.read_csv(summary_paths["by_hour"])
by_month = pd.read_csv(summary_paths["by_month"])

overall, by_hour.head(), by_month


## Optional: CatBoost reference run

Run this only after the IDW rerun has good coverage. Start with `group_kfold`


In [ ]:
cat_config = owsref.ReferenceModelConfig(
    city=city,
    output_subdir="ows_reference",
    run_label="catboost_corrected_ows_metadata_iteration01",
    calibration_mode="time_train",
    save_both_calibration_modes=True,
    run_catboost_regressor=True,
    reference_method="catboost",
    catboost_cv_mode="group_kfold",
    catboost_n_splits=5,
    catboost_iterations=1200,
    catboost_learning_rate=0.04,
    catboost_depth=8,
    catboost_thread_count=15,
    catboost_used_ram_limit="42gb",
    k_nearest_ows=8,
    ows_radius_km=8.0,
    rebuild=False,
)

cat_manifest = owsref.run_reference_pipeline(
    cat_config,
    cfm=cfm,
    cfp=cfp,
    project_dir=cwd_project,
    raw=raw,
    env_raw=env_raw,
)
cat_manifest
